# 🛡️ Support Integrity Auditor (SIA)

**End-to-end pipeline for detecting priority mismatches in customer support tickets.**

This notebook walks through every stage of the SIA system:

| Stage | Module | Description |
|-------|--------|-------------|
| 0 | `data_prep` | Load raw CRM data, extract lead sentences, build model text |
| 1 | `feature_engg` | Text cleaning, encoding, normalization |
| 2a | `template_extractor` | Subject template → severity scoring |
| 2b | `llm_scorer` | LLM-based severity scoring (pre-computed) |
| 2c | `signal_fusion` | Weighted fusion → pseudo labels |
| 3 | `train` | LightGBM severity classifier |
| 4 | `predict` | Inference on all tickets |
| 5 | `dossier_generator` | Evidence dossiers for mismatches |

---

## 📦 Setup & Imports

In [ ]:
import sys, os
import warnings
warnings.filterwarnings('ignore')

# Ensure project root is on sys.path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import json
from pathlib import Path

# Import project config — single source of truth for all paths
from src import config as C

print(f"Data directory  : {C.DATA_DIR}")
print(f"Models directory: {C.MODELS_DIR}")
print(f"Outputs directory: {C.OUTPUTS_DIR}")
print(f"Raw CSV         : {C.RAW_CSV}")
print(f"Enhanced CSV    : {C.ENHANCED_CSV}")

In [ ]:
# ── Plot style ──
plt.rcParams.update({
    'figure.figsize': (12, 5),
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

# Color palette for severity levels
SEVERITY_COLORS = {
    'Low': '#2ecc71',
    'Medium': '#f39c12',
    'High': '#e74c3c',
    'Critical': '#8e44ad'
}
MISMATCH_COLORS = {
    'Consistent': '#2ecc71',
    'Hidden Crisis': '#e74c3c',
    'False Alarm': '#3498db'
}

---
## Stage 0 — Data Preparation

Load raw CRM tickets, extract the genuine issue sentence from padded descriptions, 
and build the structured model input text.

In [ ]:
from src.data_prep import run as run_data_prep

df_prep = run_data_prep()

In [ ]:
print(f"Shape: {df_prep.shape}")
print(f"\nColumns: {list(df_prep.columns)}")
df_prep.head(3)

In [ ]:
# Priority distribution in raw data
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

priority_counts = df_prep[C.COL_PRIORITY].value_counts()
colors = [SEVERITY_COLORS.get(p, '#95a5a6') for p in priority_counts.index]

axes[0].bar(priority_counts.index, priority_counts.values, color=colors, edgecolor='white', linewidth=0.5)
axes[0].set_title('Assigned Priority Distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(priority_counts.values):
    axes[0].text(i, v + 100, str(v), ha='center', fontweight='bold')

# Category distribution
cat_counts = df_prep[C.COL_CATEGORY].value_counts()
axes[1].barh(cat_counts.index, cat_counts.values, color='#3498db', edgecolor='white', linewidth=0.5)
axes[1].set_title('Issue Category Distribution')
axes[1].set_xlabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# Sample model_text to verify the input format
print("Sample model_text entries:")
for text in df_prep['model_text'].sample(5, random_state=C.SEED).values:
    print(f"  → {text}")

---
## Stage 1 — Feature Engineering

Clean text, encode categoricals, normalize continuous features.

In [ ]:
from src.features.feature_engg import run as run_features

df_feat = run_features()

In [ ]:
print(f"Shape: {df_feat.shape}")
print(f"\nNew features added:")
new_cols = ['clean_subject', 'clean_desc', 'combined_text', 'priority_encoded',
            'channel_encoded', 'category_encoded', 'domain_tier',
            'resolution_time_z', 'satisfaction_norm']
for col in new_cols:
    if col in df_feat.columns:
        print(f"  {col}: {df_feat[col].dtype} — sample: {df_feat[col].iloc[0]}")

In [ ]:
# Resolution time vs Priority
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for priority in C.PRIORITY_LEVELS:
    subset = df_feat[df_feat[C.COL_PRIORITY] == priority][C.COL_RES_HRS]
    axes[0].hist(subset, bins=30, alpha=0.6, label=priority, 
                 color=SEVERITY_COLORS[priority])
axes[0].set_title('Resolution Time Distribution by Priority')
axes[0].set_xlabel('Hours')
axes[0].set_ylabel('Count')
axes[0].legend()

# Mean resolution time by priority
mean_res = df_feat.groupby(C.COL_PRIORITY)[C.COL_RES_HRS].mean().reindex(C.PRIORITY_LEVELS)
colors = [SEVERITY_COLORS[p] for p in mean_res.index]
axes[1].bar(mean_res.index, mean_res.values, color=colors, edgecolor='white')
axes[1].set_title('Mean Resolution Time by Priority')
axes[1].set_ylabel('Hours')
for i, v in enumerate(mean_res.values):
    axes[1].text(i, v + 0.5, f'{v:.1f}h', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

---
## Stage 2a — Template Severity Scoring

Map subject templates to severity levels based on domain knowledge.

In [ ]:
from src.pseudo_labeling.template_extractor import run as run_templates

df_templates = run_templates()

In [ ]:
# Template severity vs assigned priority — this is where mismatches originate
template_scores = pd.read_csv(str(C.TEMPLATE_SCORES_CSV))
df_merged = df_feat.merge(template_scores[[C.COL_ID, 'template_severity']], on=C.COL_ID, how='left')

# Crosstab: assigned priority vs template severity
ct = pd.crosstab(
    df_merged[C.COL_PRIORITY], 
    df_merged['template_severity'].map({0:'Low',1:'Medium',2:'High',3:'Critical'}),
    margins=True
)
print("Priority × Template Severity Crosstab:")
ct

---
## Stage 2b — LLM Severity Scoring (Pre-computed)

LLM scores were generated separately. Load and inspect them.

In [ ]:
llm_scores_path = str(C.LLM_SCORES_CSV)

if os.path.exists(llm_scores_path):
    df_llm = pd.read_csv(llm_scores_path)
    print(f"LLM scores loaded: {len(df_llm)} rows")
    print(f"\nLLM score statistics:")
    print(df_llm['llm_score'].describe())
    
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(df_llm['llm_score'], bins=50, color='#9b59b6', edgecolor='white', alpha=0.8)
    ax.set_title('LLM Score Distribution')
    ax.set_xlabel('LLM Score')
    ax.set_ylabel('Count')
    plt.tight_layout()
    plt.show()
else:
    print(f"⚠️  LLM scores not found at {llm_scores_path}")
    print("   Signal fusion will redistribute LLM weight to template scores.")

---
## Stage 2c — Signal Fusion & Pseudo Labels

Combine template, LLM, and resolution-time signals into a single inferred severity, 
then derive mismatch labels.

In [ ]:
from src.pseudo_labeling.signal_fusion import run as run_fusion

df_pseudo = run_fusion()

In [ ]:
# Inferred severity distribution
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Inferred severity distribution
sev_counts = df_pseudo['inferred_severity'].value_counts()
sev_order = ['Low', 'Medium', 'High', 'Critical']
sev_counts = sev_counts.reindex(sev_order)
colors = [SEVERITY_COLORS[s] for s in sev_order]
axes[0].bar(sev_order, sev_counts.values, color=colors, edgecolor='white')
axes[0].set_title('Inferred Severity Distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(sev_counts.values):
    axes[0].text(i, v + 100, str(v), ha='center', fontweight='bold')

# 2. Mismatch type distribution
mm_counts = df_pseudo['mismatch_type'].value_counts()
mm_colors = [MISMATCH_COLORS.get(m, '#95a5a6') for m in mm_counts.index]
axes[1].bar(mm_counts.index, mm_counts.values, color=mm_colors, edgecolor='white')
axes[1].set_title('Mismatch Type Distribution')
axes[1].set_ylabel('Count')
for i, v in enumerate(mm_counts.values):
    axes[1].text(i, v + 100, str(v), ha='center', fontweight='bold')

# 3. Inferred score histogram
axes[2].hist(df_pseudo['inferred_score'], bins=50, color='#2c3e50', edgecolor='white', alpha=0.8)
axes[2].set_title('Inferred Score Distribution')
axes[2].set_xlabel('Score')
axes[2].set_ylabel('Count')

plt.tight_layout()
plt.show()

print(f"\nMismatch rate: {df_pseudo['mismatch_label'].mean():.2%}")

In [ ]:
# Assigned vs Inferred severity heatmap
ct = pd.crosstab(
    df_pseudo[C.COL_PRIORITY], 
    df_pseudo['inferred_severity']
).reindex(index=C.PRIORITY_LEVELS, columns=['Low', 'Medium', 'High', 'Critical'])

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(ct.values, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(len(ct.columns)))
ax.set_xticklabels(ct.columns)
ax.set_yticks(range(len(ct.index)))
ax.set_yticklabels(ct.index)
ax.set_xlabel('Inferred Severity')
ax.set_ylabel('Assigned Priority')
ax.set_title('Assigned Priority vs Inferred Severity')

for i in range(len(ct.index)):
    for j in range(len(ct.columns)):
        ax.text(j, i, str(ct.values[i, j]), ha='center', va='center', 
                fontweight='bold', color='white' if ct.values[i,j] > ct.values.max()/2 else 'black')

plt.colorbar(im, ax=ax, label='Count')
plt.tight_layout()
plt.show()

---
## Stage 3 — Model Training

Train a LightGBM classifier on TF-IDF + metadata features to predict inferred severity.
Mismatch is derived by comparing predicted severity to assigned priority.

In [ ]:
from src.model.train import run as run_train

metrics = run_train()

In [ ]:
# Verification metrics summary
thresholds = {
    'Accuracy': (metrics['accuracy'], C.VERIFY_ACCURACY),
    'Macro F1': (metrics['macro_f1'], C.VERIFY_F1),
    'Recall (Consistent)': (metrics['recall_consistent'], C.VERIFY_RECALL),
    'Recall (Mismatch)': (metrics['recall_mismatch'], C.VERIFY_RECALL),
}

fig, ax = plt.subplots(figsize=(10, 5))
names = list(thresholds.keys())
values = [v[0] for v in thresholds.values()]
targets = [v[1] for v in thresholds.values()]
colors_bar = ['#2ecc71' if v >= t else '#e74c3c' for v, t in zip(values, targets)]

bars = ax.bar(names, values, color=colors_bar, edgecolor='white', linewidth=0.5, alpha=0.85)
ax.axhline(y=C.VERIFY_ACCURACY, color='#e74c3c', linestyle='--', alpha=0.5, label=f'Min threshold')

for i, (v, t) in enumerate(zip(values, targets)):
    status = '✓ PASS' if v >= t else '✗ FAIL'
    ax.text(i, v + 0.01, f'{v:.3f}\n{status}', ha='center', fontweight='bold', fontsize=10)

ax.set_ylim(0, 1.1)
ax.set_title('Verification Metrics vs Thresholds')
ax.set_ylabel('Score')
ax.legend()
plt.tight_layout()
plt.show()

---
## Stage 4 — Inference

Run the trained model on all tickets to predict severity and derive mismatches.

In [ ]:
from src.model.predict import run as run_predict

df_preds = run_predict()

In [ ]:
# Prediction results overview
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Predicted severity
pred_sev = df_preds['predicted_severity'].value_counts().reindex(['Low', 'Medium', 'High', 'Critical'])
colors = [SEVERITY_COLORS[s] for s in pred_sev.index]
axes[0].bar(pred_sev.index, pred_sev.values, color=colors, edgecolor='white')
axes[0].set_title('Predicted Severity Distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(pred_sev.values):
    axes[0].text(i, v + 100, str(v), ha='center', fontweight='bold')

# 2. Mismatch breakdown
mm = df_preds['mismatch_type'].value_counts()
mm_colors = [MISMATCH_COLORS.get(m, '#95a5a6') for m in mm.index]
axes[1].bar(mm.index, mm.values, color=mm_colors, edgecolor='white')
axes[1].set_title('Mismatch Type Breakdown')
axes[1].set_ylabel('Count')
for i, v in enumerate(mm.values):
    axes[1].text(i, v + 100, f'{v} ({v/len(df_preds)*100:.1f}%)', ha='center', fontweight='bold')

# 3. Confidence distribution
axes[2].hist(df_preds['confidence'], bins=50, color='#2c3e50', edgecolor='white', alpha=0.8)
axes[2].set_title('Model Confidence Distribution')
axes[2].set_xlabel('Confidence')
axes[2].set_ylabel('Count')

plt.tight_layout()
plt.show()

print(f"\nTotal tickets: {len(df_preds)}")
print(f"Mismatches   : {df_preds['is_mismatch'].sum()} ({df_preds['is_mismatch'].mean():.2%})")
print(f"  Hidden Crisis: {(df_preds['mismatch_type'] == 'Hidden Crisis').sum()}")
print(f"  False Alarm  : {(df_preds['mismatch_type'] == 'False Alarm').sum()}")

In [ ]:
# Mismatch by category — which categories have the most mismatches?
mismatch_by_cat = df_preds.groupby(C.COL_CATEGORY)['is_mismatch'].agg(['sum', 'mean', 'count'])
mismatch_by_cat.columns = ['Mismatches', 'Mismatch Rate', 'Total']
mismatch_by_cat = mismatch_by_cat.sort_values('Mismatch Rate', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(mismatch_by_cat.index, mismatch_by_cat['Mismatch Rate'], 
        color='#e74c3c', edgecolor='white', alpha=0.8)
ax.set_title('Mismatch Rate by Issue Category')
ax.set_xlabel('Mismatch Rate')
for i, (idx, row) in enumerate(mismatch_by_cat.iterrows()):
    ax.text(row['Mismatch Rate'] + 0.005, i, 
            f"{row['Mismatch Rate']:.1%} ({int(row['Mismatches'])}/{int(row['Total'])})", 
            va='center', fontsize=10)
plt.tight_layout()
plt.show()

---
## Stage 5 — Dossier Generation

Generate grounded evidence dossiers for every mismatch ticket.
Each claim traces to an actual ticket field — no hallucination.

In [ ]:
from src.dossier.dossier_generator import run as run_dossiers

dossiers = run_dossiers()

In [ ]:
print(f"Total dossiers generated: {len(dossiers)}")

# Show a sample Hidden Crisis dossier
hidden_crises = [d for d in dossiers if d['mismatch_type'] == 'Hidden Crisis']
if hidden_crises:
    sample = hidden_crises[0]
    print(f"\n{'='*60}")
    print(f"SAMPLE DOSSIER — Hidden Crisis")
    print(f"{'='*60}")
    print(f"Ticket ID       : {sample['ticket_id']}")
    print(f"Assigned        : {sample['assigned_priority']}")
    print(f"Inferred        : {sample['inferred_severity']}")
    print(f"Confidence      : {sample['confidence']:.4f}")
    print(f"Severity Delta  : {sample['severity_delta']}")
    print(f"\nConstraint Analysis:")
    print(f"  {sample['constraint_analysis']}")
    print(f"\nEvidence:")
    for ev in sample['feature_evidence']:
        print(f"  [{ev['signal']}] {ev.get('value', '')} → {ev.get('weight', ev.get('interpretation', ''))} (from {ev['source_field']})")

In [ ]:
# Show a sample False Alarm dossier
false_alarms = [d for d in dossiers if d['mismatch_type'] == 'False Alarm']
if false_alarms:
    sample = false_alarms[0]
    print(f"{'='*60}")
    print(f"SAMPLE DOSSIER — False Alarm")
    print(f"{'='*60}")
    print(f"Ticket ID       : {sample['ticket_id']}")
    print(f"Assigned        : {sample['assigned_priority']}")
    print(f"Inferred        : {sample['inferred_severity']}")
    print(f"Confidence      : {sample['confidence']:.4f}")
    print(f"Severity Delta  : {sample['severity_delta']}")
    print(f"\nConstraint Analysis:")
    print(f"  {sample['constraint_analysis']}")
    print(f"\nEvidence:")
    for ev in sample['feature_evidence']:
        print(f"  [{ev['signal']}] {ev.get('value', '')} → {ev.get('weight', ev.get('interpretation', ''))} (from {ev['source_field']})")

In [ ]:
# Dossier statistics
df_dossiers = pd.DataFrame(dossiers)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Mismatch types in dossiers
mt = df_dossiers['mismatch_type'].value_counts()
mt_colors = [MISMATCH_COLORS.get(m, '#95a5a6') for m in mt.index]
axes[0].pie(mt.values, labels=mt.index, colors=mt_colors, autopct='%1.1f%%',
            startangle=90, textprops={'fontsize': 12})
axes[0].set_title('Dossier Mismatch Types')

# 2. Confidence distribution for dossiers
for mt_type in ['Hidden Crisis', 'False Alarm']:
    subset = df_dossiers[df_dossiers['mismatch_type'] == mt_type]['confidence']
    if len(subset) > 0:
        axes[1].hist(subset, bins=30, alpha=0.6, label=mt_type,
                     color=MISMATCH_COLORS[mt_type])
axes[1].set_title('Model Confidence by Mismatch Type')
axes[1].set_xlabel('Confidence')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 📊 Final Summary

Key pipeline outputs and verification metrics.

In [ ]:
print("╔" + "═"*58 + "╗")
print("║" + " SUPPORT INTEGRITY AUDITOR — PIPELINE SUMMARY".ljust(58) + "║")
print("╠" + "═"*58 + "╣")
print(f"║  Total tickets processed    : {len(df_preds):>10,}" + " "*15 + "║")
print(f"║  Mismatches detected        : {df_preds['is_mismatch'].sum():>10,}" + " "*15 + "║")
print(f"║  Mismatch rate              : {df_preds['is_mismatch'].mean():>10.2%}" + " "*15 + "║")
print(f"║  Hidden Crises              : {(df_preds['mismatch_type']=='Hidden Crisis').sum():>10,}" + " "*15 + "║")
print(f"║  False Alarms               : {(df_preds['mismatch_type']=='False Alarm').sum():>10,}" + " "*15 + "║")
print(f"║  Dossiers generated         : {len(dossiers):>10,}" + " "*15 + "║")
print("╠" + "═"*58 + "╣")
print(f"║  Accuracy                   : {metrics['accuracy']:>10.4f}" + " "*15 + "║")
print(f"║  Macro F1                   : {metrics['macro_f1']:>10.4f}" + " "*15 + "║")
print(f"║  Recall (Consistent)        : {metrics['recall_consistent']:>10.4f}" + " "*15 + "║")
print(f"║  Recall (Mismatch)          : {metrics['recall_mismatch']:>10.4f}" + " "*15 + "║")
print("╠" + "═"*58 + "╣")
print("║  Output files:" + " "*43 + "║")
print(f"║    Predictions → {str(C.PREDICTIONS_CSV)[-38:]:>40}" + " ║")
print(f"║    Dossiers    → {str(C.DOSSIERS_JSON)[-38:]:>40}" + " ║")
print("╚" + "═"*58 + "╝")